In [1]:
#@title Environment
!apt-get update -qq
!apt-get install -y -qq python3.12-venv

# ============================================================
# Persistent OpenVLA environment on Google Drive
# Installs only when required versions are missing/different
# ============================================================

import os
import sys
import subprocess
import time

from google.colab import drive

# ------------------------------------------------------------
# 1. Mount Google Drive
# and Persistent Hugging Face cache
# ------------------------------------------------------------

drive.mount("/content/drive")
HF_CACHE = "/content/drive/MyDrive/huggingface_cache"

os.makedirs(HF_CACHE, exist_ok=True)

os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE

print("Hugging Face cache:", HF_CACHE)

ENV_PATH = "/content/drive/MyDrive/colab_env_openvla"
VENV_PYTHON = os.path.join(ENV_PATH, "bin", "python")

'''
if os.path.exists(ENV_PATH):
    print("Removing old/non-venv directory:")
    print(ENV_PATH)
    shutil.rmtree(ENV_PATH)
'''

print("Creating fresh persistent venv...")

# ------------------------------------------------------------
# 2. Versions we require
# ------------------------------------------------------------

REQUIRED = {
    "transformers": "4.40.1",
    "tokenizers": "0.19.1",
    "accelerate": "0.26.0",
    "timm": "0.9.10",
    "peft": "0.11.1",
    "sentencepiece": "0.1.99",
    "draccus": "0.8.0",
}

# ------------------------------------------------------------
# 3. Create persistent venv if necessary
# ------------------------------------------------------------

if not os.path.exists(VENV_PYTHON):

    print("================================================")
    print("Creating persistent OpenVLA environment")
    print("================================================")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "venv",
            "--system-site-packages",
            ENV_PATH,
        ],
        check=True,
    )

    print("✅ Virtual environment created")
else:
    print("✅ Persistent environment already exists")


# ------------------------------------------------------------
# 4. Check installed versions INSIDE the venv
# ------------------------------------------------------------

def get_installed_version(package):
    """
    Ask the persistent venv's Python which version is installed.
    """
    result = subprocess.run(
        [
            VENV_PYTHON,
            "-c",
            (
                "import importlib.metadata as m; "
                f"print(m.version('{package}'))"
            ),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        return None

    return result.stdout.strip()


print("\n================================================")
print("Checking OpenVLA environment")
print("================================================")

needs_install = []

for package, required_version in REQUIRED.items():

    installed = get_installed_version(package)

    if installed is None:
        print(f"❌ {package}: NOT INSTALLED")
        needs_install.append(package)

    elif required_version is None:
        print(f"✅ {package}: {installed}")

    elif installed != required_version:
        print(
            f"⚠️ {package}: {installed} "
            f"(required {required_version})"
        )
        needs_install.append(package)

    else:
        print(f"✅ {package}: {installed}")


# ------------------------------------------------------------
# 5. Install ONLY if something is missing/wrong
# ------------------------------------------------------------

if needs_install:

    print("\n================================================")
    print("Installing/updating OpenVLA dependencies")
    print("================================================")

    packages = [
        "transformers==4.40.1",
        "tokenizers==0.19.1",
        "accelerate==0.26.0",
        "timm==0.9.10",
        "peft==0.11.1",
        "sentencepiece==0.1.99",
        "draccus==0.8.0",
    ]

    subprocess.run(
        [
            VENV_PYTHON,
            "-m",
            "pip",
            "install",
            "-q",
            *packages,
        ],
        check=True,
    )

    print("✅ Installation/update complete")

else:

    print("\n================================================")
    print("🎉 OpenVLA environment already satisfies all")
    print("   required versions.")
    print("   SKIPPING pip installation.")
    print("================================================")


# ------------------------------------------------------------
# 6. Make the venv packages available to THIS notebook
# ------------------------------------------------------------

# We are using Colab's Python kernel, so add the venv's
# site-packages to sys.path.

#PYTHON_VERSION = f"{sys.version_info.major}.{sys.version_info.minor}"
PYTHON_VERSION = subprocess.check_output(
    [
        VENV_PYTHON,
        "-c",
        "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')",
    ],
    text=True,
).strip()

VENV_SITE = os.path.join(
    ENV_PATH,
    "lib",
    f"python{PYTHON_VERSION}",
    "site-packages",
)

if os.path.exists(VENV_SITE):

    if VENV_SITE not in sys.path:
        sys.path.insert(0, VENV_SITE)

    print("\nVenv site-packages:")
    print(VENV_SITE)

else:

    print("\n⚠️ WARNING:")
    print("Could not find:")
    print(VENV_SITE)

print("Notebook Python:")
print(sys.version)
print("\nNotebook executable:")
print(sys.executable)

# ------------------------------------------------------------
# 8. Install robotics / simulation packages INTO THE VENV
# ------------------------------------------------------------

print("\n================================================")
print("ROBOTICS ENVIRONMENT")
print("================================================")

ROBOTICS_REQUIRED = {
    "mujoco": "3.3.1",
    "mediapy": None,
    "robosuite": "1.4.1",
}

def get_installed_version(package):
    result = subprocess.run(
        [
            VENV_PYTHON,
            "-c",
            (
                "import importlib.metadata as m; "
                f"print(m.version('{package}'))"
            ),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        return None

    return result.stdout.strip()


for package, required_version in ROBOTICS_REQUIRED.items():

    installed = get_installed_version(package)

    if installed is None:
        print(f"❌ {package}: NOT INSTALLED")

        if required_version is None:
            spec = package
        else:
            spec = f"{package}=={required_version}"

        subprocess.run(
            [
                VENV_PYTHON,
                "-m",
                "pip",
                "install",
                "-q",
                spec,
            ],
            check=True,
        )

        print(f"✅ Installed {spec}")

    elif required_version is not None and installed != required_version:
        print(
            f"⚠️ {package}: {installed} "
            f"(required {required_version})"
        )

        subprocess.run(
            [
                VENV_PYTHON,
                "-m",
                "pip",
                "install",
                "-q",
                "--force-reinstall",
                f"{package}=={required_version}",
            ],
            check=True,
        )

        print(f"✅ Changed {package} to {required_version}")

    else:
        print(f"✅ {package}: {installed}")

# ------------------------------------------------------------
# 9. Verify robotics environment and OpenVLA using VENV Python
# ------------------------------------------------------------

'''
print("\n================================================")
print("ROBOTICS and OPENVLA VERIFICATION")
print("================================================")

def check_import(name, import_code):
    print("=" * 60)
    print(f"Testing {name}...")
    print("=" * 60)

    start = time.time()

    result = subprocess.run(
        [
            VENV_PYTHON,
            "-c",
            import_code,
        ],
        text=True,
    )

    elapsed = time.time() - start
    print(f"Time: {elapsed:.2f}s", flush=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"{name} verification failed."
        )

    print(f"{name}: OK\n", flush=True)


# Python
check_import(
    "Python",
    """
import sys
print("Python executable:", sys.executable, flush=True)
print("Python version   :", sys.version.split()[0], flush=True)
"""
)

imports = [
    "torch",
    "transformers",
    "timm",
    "accelerate",
    "draccus",
    "mujoco",
    "robosuite",
    "mediapy"
]

for package in imports:
    print(f"\n>>> Testing {package}...", flush=True)

    t = time.time()

    p = subprocess.run(
        [
            VENV_PYTHON,
            "-c",
            f"import {package}; print('OK', flush=True)"
        ],

        timeout=120,
    )

    print(f">>> {package}: {time.time() - t:.2f}s", flush=True)

print("=== PEFT package info ===")

subprocess.run(
    [
        VENV_PYTHON,
        "-c",
        "import importlib.metadata; print(importlib.metadata.version('peft'), flush=True)"
    ],
    text=True,
    timeout=30,
)


print("================================================")
print("=== OPENVLA + ROBOSUITE ENVIRONMENT OK ===")
print("================================================")
'''
print("\n✅ Persistent OpenVLA environment ready!!!")


W: An error occurred during the signature verification. The repository is not updated and the previous index files will be used. GPG error: https://cli.github.com/packages stable InRelease: The following signatures were invalid: EXPKEYSIG 23F3D4EA75716059 GitHub CLI <opensource+cli@github.com> The following signatures couldn't be verified because the public key is not available: NO_PUBKEY 5612B36462313325
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://cli.github.com/packages/dists/stable/InRelease  The following signatures were invalid: EXPKEYSIG 23F3D4EA75716059 GitHub CLI <opensource+cli@github.com> The following signatures couldn't be verified because the public key is not available: NO_PUBKEY 5612B36462313325
W: Some index files failed to download. They have been ignored, or old ones used instead.
Selecting previous

In [ ]:
#@title FlashAttn
import subprocess
import sys
FLASH_ATTN_VERSION = "2.5.5"
#VENV_PYTHON = "/content/drive/MyDrive/colab_env_openvla/bin/python"

for pkg in ["packaging", "ninja"]:
    print(f"\nChecking {pkg}...", flush=True)

    result = subprocess.run(
        [
            VENV_PYTHON,
            "-c",
            f"""
import importlib.metadata
print(importlib.metadata.version("{pkg}"))
"""
        ],
        capture_output=True,
        text=True,
    )

    print("Return code:", result.returncode)
    print("Version:", result.stdout.strip())

    if result.stderr.strip():
        print("stderr:", result.stderr.strip())



# -------------------------------------------------------------------------
# Flash Attention
# -------------------------------------------------------------------------

def python_import_test(module):
    r = subprocess.run(
        [
            VENV_PYTHON,
            "-c",
            f"import {module}",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env={
            **os.environ,
            "MPLBACKEND": "Agg",
            "MUJOCO_GL": "egl",
        },
    )
    return r.returncode == 0

def run(cmd, cwd=None, env=None, input_data=None, timeout=600, check=True):
    """Run command and print complete output."""

    print("\n$ " + " ".join(str(x) for x in cmd))
    print("-" * 72)

    r = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        input=input_data,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )

    if r.stdout:
        print(r.stdout)

    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {r.returncode}:\n"
            + " ".join(str(x) for x in cmd)
        )

    return r

if not python_import_test("flash_attn"):

    print("\nFlash Attention missing — installing 2.5.5...")

    run([
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "flash-attn==2.5.5",
        "--no-build-isolation",
    ])

else:

    print("Flash Attention already imports.")




Checking packaging...
Return code: 0
Version: 26.0

Checking ninja...
Return code: 0
Version: 1.13.2

Flash Attention missing — installing 2.5.5...

$ /content/drive/MyDrive/colab_env_openvla/bin/python -m pip install flash-attn==2.5.5 --no-build-isolation
------------------------------------------------------------------------


In [ ]:
#@title Inference
# ============================================================
# OpenVLA + Robosuite Panda Inference
#
# Pipeline:
#
#   Robosuite camera
#        ↓
#   image preprocessing
#        ↓
#   OpenVLA 7B (base model)
#        ↓
#   bridge_orig 7-D action
#        ↓
#   Robosuite OSC_POSE controller
#        ↓
#   Panda
#
# Action:
#   [dx, dy, dz, dRx, dRy, dRz, gripper]
#
# The robot controller / IK is handled by Robosuite.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import time
import math
import imageio

import numpy as np
import torch

from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

import robosuite as suite
from robosuite.controllers import load_controller_config


# ============================================================
# 2. CONFIGURATION
# ============================================================

class Config:

    # --------------------------------------------------------
    # OpenVLA
    # --------------------------------------------------------

    model_name = "openvla/openvla-7b"

    # IMPORTANT:
    # This is the action normalization key used by the
    # general/base OpenVLA model for BridgeData.
    normalization_key = "bridge_orig"

    gpu = "cuda:0"

    # --------------------------------------------------------
    # Task
    # --------------------------------------------------------

    instruction = "pick up the red cube"

    # --------------------------------------------------------
    # Camera
    # --------------------------------------------------------

    camera_name = "agentview"

    camera_height = 224
    camera_width = 224

    # --------------------------------------------------------
    # Simulation
    # --------------------------------------------------------

    control_freq = 20

    maximum_steps = 300

    # Let the simulation settle before asking OpenVLA
    # for the first action.
    warmup_steps = 20

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    output_folder = "/content/PickAgent/outputs/videos"

    episode_number = 0

    # --------------------------------------------------------
    # Action safety
    # --------------------------------------------------------

    # Number of repeated controller steps per VLA action.
    #
    # Start with 1.
    #
    # If movement is too slow, this can be increased later.
    action_repeat = 1

    # --------------------------------------------------------
    # Debug
    # --------------------------------------------------------

    print_actions = True
    print_robot_state = True


# ============================================================
# 3. RUNTIME ENVIRONMENT
# ============================================================

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# We are running this script with:
#
# /content/openvla_env/bin/python
#
# so the following paths are useful when running the script
# directly from PickAgent.
#
# They are harmless if the paths already exist in sys.path.

import sys

PROJECT_PATHS = [
    "/content/PickAgent",
    "/content/openvla",
]

for path in PROJECT_PATHS:
    if path not in sys.path:
        sys.path.insert(0, path)


# ============================================================
# 4. OPENVLA
# ============================================================

class OpenVLA:
    """
    Small wrapper around the general OpenVLA model.
    """

    def __init__(
        self,
        model_name,
        gpu="cuda:0",
        normalization_key="bridge_orig",
    ):

        self.gpu = gpu
        self.normalization_key = normalization_key

        print()
        print("=" * 70)
        print("Loading OpenVLA")
        print("=" * 70)

        print("Model:", model_name)
        print("GPU:", gpu)
        print("Normalization:", normalization_key)

        # ----------------------------------------------------
        # Processor
        # ----------------------------------------------------

        print()
        print("Loading processor...")

        self.processor = AutoProcessor.from_pretrained(
            model_name,
            trust_remote_code=True,
        )

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        print("Loading model...")

        self.model = AutoModelForVision2Seq.from_pretrained(
            model_name,

            # Use Flash Attention when available.
            attn_implementation="flash_attention_2",

            torch_dtype=torch.bfloat16,

            low_cpu_mem_usage=True,

            trust_remote_code=True,
        ).to(gpu)

        self.model.eval()

        print()
        print("OpenVLA loaded successfully.")

        print(
            "Model dtype:",
            next(self.model.parameters()).dtype,
        )

        print(
            "Model device:",
            next(self.model.parameters()).device,
        )

        print("=" * 70)
        print()


    @torch.inference_mode()
    def predict(
        self,
        image,
        instruction,
    ):
        """
        Predict one OpenVLA action.

        Returns:
            np.ndarray with shape (7,)
        """

        # ----------------------------------------------------
        # Convert image to PIL
        # ----------------------------------------------------

        if isinstance(image, np.ndarray):

            image = Image.fromarray(
                image
            ).convert("RGB")

        elif not isinstance(image, Image.Image):

            raise TypeError(
                f"Unsupported image type: {type(image)}"
            )

        # ----------------------------------------------------
        # OpenVLA prompt
        # ----------------------------------------------------

        prompt = (
            "In: What action should the robot take to "
            f"{instruction.lower()}?\nOut:"
        )

        # ----------------------------------------------------
        # Processor
        # ----------------------------------------------------

        inputs = self.processor(
            prompt,
            image,
        )

        # Move tensors to GPU.
        #
        # We do NOT blindly convert every object to bfloat16.
        # Integer tensors such as input_ids must remain integer.
        #
        inputs = {
            key: value.to(self.gpu)
            if hasattr(value, "to")
            else value
            for key, value in inputs.items()
        }

        # ----------------------------------------------------
        # Predict
        # ----------------------------------------------------

        action = self.model.predict_action(
            **inputs,

            unnorm_key=self.normalization_key,

            do_sample=False,
        )

        # ----------------------------------------------------
        # Normalize output type
        # ----------------------------------------------------

        if isinstance(action, torch.Tensor):

            action = action.detach().float().cpu().numpy()

        else:

            action = np.asarray(action)

        action = np.asarray(
            action,
            dtype=np.float32,
        ).reshape(-1)

        if action.shape[0] != 7:

            raise RuntimeError(
                "OpenVLA returned an unexpected action shape: "
                f"{action.shape}. Expected 7 values."
            )

        return action


# ============================================================
# 5. ROBOSUITE ENVIRONMENT
# ============================================================

def create_environment():
    """
    Create a Robosuite Panda environment.

    OSC_POSE receives a normalized 7-D action:

        [dx, dy, dz, dRx, dRy, dRz, gripper]

    Robosuite performs the robot-level control internally.
    """

    print()
    print("=" * 70)
    print("Creating Robosuite environment")
    print("=" * 70)

    # --------------------------------------------------------
    # Controller
    # --------------------------------------------------------

    controller_config = load_controller_config(
        default_controller="OSC_POSE"
    )

    print()
    print("Controller:")
    print("  OSC_POSE")

    # --------------------------------------------------------
    # Environment
    # --------------------------------------------------------

    env = suite.make(
        "Lift",

        robots="Panda",

        controller_configs=controller_config,

        # ----------------------------------------------------
        # Rendering
        # ----------------------------------------------------

        has_renderer=False,

        has_offscreen_renderer=True,

        render_camera=Config.camera_name,

        # ----------------------------------------------------
        # Camera observations
        # ----------------------------------------------------

        use_camera_obs=True,

        camera_names=Config.camera_name,

        camera_heights=Config.camera_height,

        camera_widths=Config.camera_width,

        # ----------------------------------------------------
        # Simulation
        # ----------------------------------------------------

        control_freq=Config.control_freq,

        horizon=Config.maximum_steps,

        # Reward is useful for diagnostics.
        reward_shaping=True,

        # ----------------------------------------------------
        # Determinism
        # ----------------------------------------------------

        ignore_done=False,
    )

    print()
    print("Robosuite environment created successfully.")

    print("=" * 70)
    print()

    return env


# ============================================================
# 6. IMAGE PROCESSING
# ============================================================

def prepare_image(observation):
    """
    Convert Robosuite's camera observation into the image
    expected by OpenVLA.

    IMPORTANT:

    Robosuite camera observations are vertically flipped
    relative to the normal image convention.

    Therefore we flip Y only:

        image[::-1]

    We do NOT rotate 180 degrees like the LIBERO code.
    """

    if Config.camera_name not in observation:

        raise RuntimeError(
            f"Camera '{Config.camera_name}' not found in "
            f"observation keys: {list(observation.keys())}"
        )

    image = observation[
        Config.camera_name
    ]

    image = np.asarray(
        image,
        dtype=np.uint8,
    )

    # --------------------------------------------------------
    # Robosuite / MuJoCo camera convention
    # --------------------------------------------------------

    image = image[::-1]

    # --------------------------------------------------------
    # Ensure RGB
    # --------------------------------------------------------

    if image.ndim != 3:

        raise RuntimeError(
            f"Unexpected camera image shape: {image.shape}"
        )

    if image.shape[2] != 3:

        raise RuntimeError(
            f"Expected RGB image, got: {image.shape}"
        )

    return image


# ============================================================
# 7. OPENVLA → ROBOSUITE ACTION
# ============================================================

def convert_vla_action_to_robosuite(
    vla_action,
):
    """
    Convert OpenVLA's 7-D BridgeData action to the action
    expected by the Robosuite OSC_POSE controller.

    OpenVLA:

        [dx, dy, dz, dRx, dRy, dRz, gripper]

    Robosuite OSC_POSE:

        [dx, dy, dz, dRx, dRy, dRz, gripper]

    The important point is that Robosuite owns the
    robot-level IK / OSC computation.

    We therefore DO NOT calculate a Jacobian here.

    --------------------------------------------------------

    Gripper:

        OpenVLA:
            0 = open
            1 = closed

        Robosuite:
            -1 = open
            +1 = closed

    """

    action = np.asarray(
        vla_action,
        dtype=np.float32,
    ).copy()

    if action.shape != (7,):

        action = action.reshape(-1)

        if action.shape[0] != 7:

            raise RuntimeError(
                f"Expected 7-D action, got {action.shape}"
            )

    # --------------------------------------------------------
    # Copy the Cartesian pose action.
    # --------------------------------------------------------

    robosuite_action = action.copy()

    # --------------------------------------------------------
    # Gripper conversion
    # --------------------------------------------------------

    # OpenVLA:
    #
    #     0 -> open
    #     1 -> closed
    #
    # Robosuite:
    #
    #     -1 -> open
    #     +1 -> closed

    robosuite_action[6] = (
        2.0 * action[6] - 1.0
    )

    # --------------------------------------------------------
    # Clamp gripper.
    # --------------------------------------------------------

    robosuite_action[6] = np.clip(
        robosuite_action[6],
        -1.0,
        1.0,
    )

    return robosuite_action


# ============================================================
# 8. ROBOT STATE DIAGNOSTICS
# ============================================================

def print_robot_state(
    env,
    step_number,
):
    """
    Print the Panda end-effector state.

    This is diagnostic only.
    """

    if not Config.print_robot_state:
        return

    try:

        robot = env.robots[0]

        # End-effector position
        eef_pos = np.asarray(
            robot.sim.data.site_xpos[
                robot.eef_site_id
            ]
        )

        # End-effector orientation
        eef_rot = np.asarray(
            robot.sim.data.site_xmat[
                robot.eef_site_id
            ]
        ).reshape(3, 3)

        print(
            f"  EEF position: "
            f"[{eef_pos[0]:+.4f}, "
            f"{eef_pos[1]:+.4f}, "
            f"{eef_pos[2]:+.4f}]"
        )

        print(
            "  EEF rotation matrix:"
        )

        print(eef_rot)

    except Exception as exc:

        print(
            "  Could not read EEF state:",
            repr(exc),
        )


# ============================================================
# 9. VIDEO
# ============================================================

def save_video(
    frames,
    instruction,
    episode_number,
):
    """
    Save recorded RGB frames as MP4.
    """

    os.makedirs(
        Config.output_folder,
        exist_ok=True,
    )

    safe_instruction = (
        instruction
        .lower()
        .replace(" ", "_")
        .replace("\n", "_")
        .replace(".", "_")
    )[:60]

    video_file = os.path.join(
        Config.output_folder,
        (
            f"episode={episode_number}"
            f"--prompt={safe_instruction}.mp4"
        ),
    )

    print()
    print("Saving video:")
    print(video_file)

    writer = imageio.get_writer(
        video_file,
        fps=30,
    )

    try:

        for frame in frames:

            writer.append_data(frame)

    finally:

        writer.close()

    print("Video saved.")

    return video_file


# ============================================================
# 10. SINGLE EPISODE
# ============================================================

def run_episode(
    env,
    vla,
    instruction,
):
    """
    Run one complete OpenVLA → Robosuite episode.
    """

    print()
    print("=" * 70)
    print("STARTING EPISODE")
    print("=" * 70)

    print()
    print("Instruction:")
    print(instruction)

    # --------------------------------------------------------
    # Reset
    # --------------------------------------------------------

    observation = env.reset()

    frames = []

    successful = False

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    print()
    print(
        f"Warming up for {Config.warmup_steps} steps..."
    )

    # Robosuite action dimension:
    #
    # [dx,dy,dz,dRx,dRy,dRz,gripper]

    no_op = np.zeros(
        7,
        dtype=np.float32,
    )

    # Gripper open during warmup.
    no_op[6] = -1.0

    for warmup_step in range(
        Config.warmup_steps
    ):

        observation, reward, done, info = env.step(
            no_op
        )

    print("Warm-up complete.")

    # --------------------------------------------------------
    # Main loop
    # --------------------------------------------------------

    for step_number in range(
        Config.maximum_steps
    ):

        print()
        print("-" * 70)
        print(
            f"STEP {step_number + 1} / "
            f"{Config.maximum_steps}"
        )
        print("-" * 70)

        # ----------------------------------------------------
        # Image
        # ----------------------------------------------------

        camera_frame = prepare_image(
            observation
        )

        frames.append(
            camera_frame.copy()
        )

        if step_number == 0:

            print()
            print(
                "Camera image shape:",
                camera_frame.shape,
            )

            print(
                "Camera image dtype:",
                camera_frame.dtype,
            )

        # ----------------------------------------------------
        # OpenVLA
        # ----------------------------------------------------

        t0 = time.time()

        vla_action = vla.predict(
            image=camera_frame,
            instruction=instruction,
        )

        inference_time = (
            time.time() - t0
        )

        # ----------------------------------------------------
        # Convert action
        # ----------------------------------------------------

        robosuite_action = (
            convert_vla_action_to_robosuite(
                vla_action
            )
        )

        # ----------------------------------------------------
        # Diagnostics
        # ----------------------------------------------------

        if Config.print_actions:

            print()
            print(
                "OpenVLA action:"
            )

            print(
                np.array2string(
                    vla_action,
                    precision=5,
                    suppress_small=False,
                )
            )

            print()
            print(
                "Robosuite action:"
            )

            print(
                np.array2string(
                    robosuite_action,
                    precision=5,
                    suppress_small=False,
                )
            )

            print()
            print(
                f"Inference time: "
                f"{inference_time:.3f} sec"
            )

        # ----------------------------------------------------
        # Execute action
        # ----------------------------------------------------

        for repeat_number in range(
            Config.action_repeat
        ):

            (
                observation,
                reward,
                done,
                info,
            ) = env.step(
                robosuite_action
            )

        # ----------------------------------------------------
        # State diagnostics
        # ----------------------------------------------------

        print_robot_state(
            env,
            step_number,
        )

        # ----------------------------------------------------
        # Reward
        # ----------------------------------------------------

        print(
            f"  Reward: {reward}"
        )

        # ----------------------------------------------------
        # Success
        # ----------------------------------------------------

        # Robosuite's Lift environment provides reward/success
        # information through its reward / info machinery.
        #
        # We check both where possible.

        is_success = False

        if isinstance(info, dict):

            is_success = bool(
                info.get(
                    "success",
                    False,
                )
            )

        if not is_success:

            try:

                is_success = bool(
                    env._check_success()
                )

            except Exception:

                pass

        if is_success:

            successful = True

            print()
            print("=" * 70)
            print(
                "SUCCESS: Robosuite reports task success."
            )
            print("=" * 70)

            break

        if done:

            print()
            print(
                "Environment reported done."
            )

            break

    # --------------------------------------------------------
    # Failure
    # --------------------------------------------------------

    if not successful:

        print()
        print("=" * 70)
        print(
            "FAILED / TIMEOUT: "
            "Task was not completed."
        )
        print("=" * 70)

    # --------------------------------------------------------
    # Save video
    # --------------------------------------------------------

    video_file = save_video(
        frames=frames,
        instruction=instruction,
        episode_number=Config.episode_number,
    )

    return successful, video_file


# ============================================================
# 11. MAIN
# ============================================================

def main():

    print()
    print("=" * 70)
    print("OPENVLA + ROBOSUITE PANDA")
    print("=" * 70)

    print()
    print("Python:")
    print(sys.executable)

    print()
    print("PyTorch:")
    print(torch.__version__)

    print()
    print("CUDA available:")
    print(torch.cuda.is_available())

    if torch.cuda.is_available():

        print(
            "GPU:",
            torch.cuda.get_device_name(0),
        )

    # --------------------------------------------------------
    # Create VLA
    # --------------------------------------------------------

    vla = OpenVLA(
        model_name=Config.model_name,
        gpu=Config.gpu,
        normalization_key=Config.normalization_key,
    )

    # --------------------------------------------------------
    # Create Robosuite
    # --------------------------------------------------------

    env = create_environment()

    try:

        # ----------------------------------------------------
        # Run
        # ----------------------------------------------------

        successful, video_file = run_episode(
            env=env,
            vla=vla,
            instruction=Config.instruction,
        )

        print()
        print("=" * 70)
        print("SIMULATION COMPLETE")
        print("=" * 70)

        print()
        print(
            "Result:",
            "SUCCESS" if successful else "FAILURE",
        )

        print()
        print(
            "Video:",
            video_file,
        )

    finally:

        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        print()
        print("Closing Robosuite environment...")

        env.close()

        print("Environment closed.")


# ============================================================
# 12. RUN
# ============================================================

#if __name__ == "__main__":
#    main()

vla = OpenVLA(
    model_name=Config.model_name,
    gpu=Config.gpu,
    normalization_key=Config.normalization_key,
)

env = create_environment()

In [ ]:
#@title Single Action Diagnostic
# ============================================================
# OPENVLA + ROBOSUITE SINGLE-ACTION DIAGNOSTIC
# ============================================================

# ------------------------------------------------------------
# Reset
# ------------------------------------------------------------

observation = env.reset()

print("\nInitial observation keys:")
print(list(observation.keys()))

# ------------------------------------------------------------
# Camera
# ------------------------------------------------------------

camera_frame = prepare_image(observation)

print("\nCamera frame:")
print("  shape :", camera_frame.shape)
print("  dtype :", camera_frame.dtype)
print("  min   :", camera_frame.min())
print("  max   :", camera_frame.max())

# ------------------------------------------------------------
# OpenVLA inference
# ------------------------------------------------------------

print("\nRunning OpenVLA inference...")

t0 = time.time()

vla_action = vla.predict(
    image=camera_frame,
    instruction=Config.instruction,
)

inference_time = time.time() - t0

# ------------------------------------------------------------
# Convert
# ------------------------------------------------------------

robosuite_action = convert_vla_action_to_robosuite(
    vla_action
)

# ------------------------------------------------------------
# Print action
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACTION DIAGNOSTIC")
print("=" * 70)

print("\nInstruction:")
print(Config.instruction)

print("\nOpenVLA raw action:")
print(
    np.array2string(
        vla_action,
        precision=7,
        suppress_small=False,
    )
)

print("\nRobosuite action:")
print(
    np.array2string(
        robosuite_action,
        precision=7,
        suppress_small=False,
    )
)

print("\nRobosuite action spec:")
print(env.action_spec)

print("\nInference time:")
print(f"{inference_time:.3f} sec")

# ------------------------------------------------------------
# EEF before
# ------------------------------------------------------------

eef_before = observation["robot0_eef_pos"].copy()

print("\nEEF before:")
print(eef_before)

# ------------------------------------------------------------
# Execute EXACTLY ONE action
# ------------------------------------------------------------

print("\nExecuting ONE Robosuite step...")

observation, reward, done, info = env.step(
    robosuite_action
)

# ------------------------------------------------------------
# EEF after
# ------------------------------------------------------------

eef_after = observation["robot0_eef_pos"].copy()

print("\nEEF after:")
print(eef_after)

print("\nEEF displacement:")
print(eef_after - eef_before)

print("\nReward:")
print(reward)

print("\nDone:")
print(done)

print("\nInfo:")
print(info)

print("=" * 70)
print("SINGLE-ACTION DIAGNOSTIC COMPLETE")
print("=" * 70)